# 06 — X-IIoTID Ensemble Development

**Goal:** Check whether combining LightGBM, Decision Tree and Random Forest improves attack detection.

**Input:** Saved V3 models and the **full frozen validation set** (123,125 rows).

**Output:** Model-error comparison, ensemble scores, resource measurements and a small set of graphs.

**Rule:** No retraining, no new data split, and **no test-set access**. Results are exploratory validation evidence.

## 1. Setup

Load only the three shortlisted V3 models and the frozen validation data.

In [ ]:
from pathlib import Path
from time import perf_counter
from datetime import datetime, timezone
from threading import Event, Thread
from itertools import combinations
import os, gc, json, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib, psutil
from IPython.display import display
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, accuracy_score

ROOT = Path(r'D:\J26-CS-336-Network-Flow-V2\network_flow_v2')
VALID_FILE = ROOT / 'data/splits/x_iiotid/x_iiotid_core19_validation_v3.csv.gz'
SPLIT_REPORT = ROOT / 'reports/audits/x_iiotid_v3_split_report.json'
MODEL_DIR = ROOT / 'models/x_iiotid_v3'
AUDITS = ROOT / 'reports/audits'
METRICS = ROOT / 'reports/metrics'
FIGURES = ROOT / 'reports/figures'
SHORTLIST = ('lightgbm','decision_tree','random_forest')
EXPECTED = {'train':574585,'validation':123125,'test':123124}
CAT = ['Protocol','Service','Conn_state','is_syn_only','Is_SYN_ACK',
       'is_pure_ack','is_with_payload','FIN or RST']
MISSING = {'','-','?','na','n/a','nan','none','null','missing','unknown','<missing>'}
for path in (VALID_FILE,SPLIT_REPORT):
    if not path.is_file(): raise FileNotFoundError(f'Missing Notebook 03 output: {path}')
split = json.loads(SPLIT_REPORT.read_text(encoding='utf-8'))
if {k:int(v) for k,v in split['counts'].items()} != EXPECTED:
    raise ValueError('Frozen split counts have changed')
if split.get('normalized_signature_overlap_groups') != 0:
    raise ValueError('Normalized feature signatures overlap between partitions')
if split.get('test_scored') is True:
    raise ValueError('Test was previously scored; investigate before proceeding')
FEATURES = list(split['features'])
NUM = [c for c in FEATURES if c not in CAT]
if len(FEATURES)!=19 or len(NUM)!=11 or not set(CAT).issubset(FEATURES):
    raise ValueError('Unexpected selected features')
for folder in (AUDITS,METRICS,FIGURES): folder.mkdir(parents=True,exist_ok=True)
print('Dataset: X-IIoTID | Features: 19 | Validation: 123,125 rows | TEST NOT OPENED')

## 2. Prepare validation inputs

Use the same feature cleaning as Notebook 04. The labels are only for checking predictions.

In [ ]:
frame = pd.read_csv(VALID_FILE,dtype=str,keep_default_na=False,low_memory=False)
expected_columns = ['csv_row_index_0based']+FEATURES+['class1','class2','class3']
if frame.columns.tolist()!=expected_columns or len(frame)!=EXPECTED['validation']:
    raise ValueError('Validation data does not match Notebook 03 outputs')
if frame['csv_row_index_0based'].duplicated().any():
    raise ValueError('Duplicate validation row IDs')
if not frame['class3'].isin(['Normal','Attack']).all():
    raise ValueError('Unexpected binary labels')

def clean_features(source):
    out=source[FEATURES].copy()
    for column in NUM:
        s=out[column].astype(str).str.strip()
        s=s.mask(s.str.casefold().isin(MISSING))
        out[column]=pd.to_numeric(s,errors='coerce').replace([np.inf,-np.inf],np.nan).astype(np.float32)
    for column in CAT:
        s=out[column].astype(str).str.strip()
        out[column]=s.mask(s.str.casefold().isin(MISSING),'__MISSING__')
    return out

X = clean_features(frame)
y = (frame['class3']=='Attack').to_numpy(dtype=np.uint8)
attack = (y==1)
normal = ~attack
print(f'Validation records: {len(y):,} | Normal: {normal.sum():,} | Attack: {attack.sum():,}')

## 3. Load models and verify predictions

Each saved model must reproduce its Notebook 04 confusion-matrix counts.

In [ ]:
models, predictions, probabilities, source_metrics = {}, {}, {}, {}
for name in SHORTLIST:
    file = MODEL_DIR / f'{name}_core19_v3.joblib'
    audit = AUDITS / f'x_iiotid_v3_{name}_training.json'
    for path in (file,audit):
        if not path.is_file(): raise FileNotFoundError(f'Run Notebook 04 first: {path}')
    model=joblib.load(file)
    if list(model.classes_) != [0,1]:
        raise ValueError(f'{name}: unexpected class ordering: {model.classes_}')
    p=np.asarray(model.predict(X),dtype=np.uint8)
    proba=np.asarray(model.predict_proba(X),dtype=float)[:,1]
    if not np.isfinite(proba).all() or len(p)!=len(y):
        raise ValueError(f'{name}: invalid prediction probabilities')
    expected=json.loads(audit.read_text(encoding='utf-8'))
    if expected.get('test_scored') is not False:
        raise ValueError(f'{name}: unexpected test-scored flag')
    tn,fp,fn,tp=[int(v) for v in confusion_matrix(y,p,labels=[0,1]).ravel()]
    expected_cm=[int(expected['metrics'][key]) for key in ('tn','fp','fn','tp')]
    if [tn,fp,fn,tp] != expected_cm:
        raise ValueError(f'{name}: saved model disagrees with Notebook 04: {[tn,fp,fn,tp]} != {expected_cm}')
    models[name]=model
    predictions[name]=p
    probabilities[name]=proba
    source_metrics[name]=expected
    print(f'{name:14s} FP={fp:,} FN={fn:,} F1={100*f1_score(y,p):.2f}%')
print('All three models verified against Notebook 04.')

## 4. Find different model mistakes

**Error diversity** means one model correctly detects an attack missed by another model. Check the exact record overlap, not just F1 scores.

In [ ]:
diversity=[]
for a,b in combinations(SHORTLIST,2):
    pa,pb=predictions[a],predictions[b]
    a_miss=attack & (pa==0)
    b_miss=attack & (pb==0)
    a_fp=normal & (pa==1)
    b_fp=normal & (pb==1)
    diversity.append({
        'first_model':a,'second_model':b,
        'both_miss_same_attacks':int((a_miss & b_miss).sum()),
        'attacks_missed_by_first_rescued_by_second':int((a_miss & (pb==1)).sum()),
        'attacks_missed_by_second_rescued_by_first':int((b_miss & (pa==1)).sum()),
        'new_false_alarms_if_second_added_by_OR':int((~a_fp & b_fp).sum()),
        'false_alarms_shared':int((a_fp & b_fp).sum())
    })
diversity_df=pd.DataFrame(diversity)
display(diversity_df)

## 5. Compare individual models and simple ensembles

**OR:** Attack if either model says Attack. **Majority:** At least two of three models say Attack. **Mean probability:** Average the three attack probabilities and use the fixed 0.5 threshold.

These are validation-only experiments. No threshold is tuned on the test set.

In [ ]:
lgb,dt,rf=(predictions[name] for name in SHORTLIST)
variants = {
    'LightGBM':lgb,
    'Decision Tree':dt,
    'Random Forest':rf,
    'LightGBM + RF (OR)':(lgb | rf).astype(np.uint8),
    'LightGBM + DT (OR)':(lgb | dt).astype(np.uint8),
    'LightGBM + RF (AND)':(lgb & rf).astype(np.uint8),
    'Three-model majority':((lgb.astype(int)+dt.astype(int)+rf.astype(int))>=2).astype(np.uint8),
    'Three-model OR':(lgb | dt | rf).astype(np.uint8),
    'Three-model mean probability':((probabilities['lightgbm']+probabilities['decision_tree']+
        probabilities['random_forest'])/3 >= 0.5).astype(np.uint8)
}

def calculate_scores(label,p):
    tn,fp,fn,tp=[int(v) for v in confusion_matrix(y,p,labels=[0,1]).ravel()]
    return {
        'candidate':label,'attack_f1_pct':round(100*float(f1_score(y,p,zero_division=0)),3),
        'attack_precision_pct':round(100*float(precision_score(y,p,zero_division=0)),3),
        'attack_recall_pct':round(100*float(recall_score(y,p,zero_division=0)),3),
        'false_positive_pct':round(100*fp/(tn+fp),3),
        'accuracy_pct':round(100*float(accuracy_score(y,p)),3),
        'false_alarms':fp,'missed_attacks':fn,'tn':tn,'fp':fp,'fn':fn,'tp':tp
    }

comparison=pd.DataFrame([calculate_scores(label,p) for label,p in variants.items()])
display(comparison[['candidate','attack_f1_pct','false_positive_pct',
                    'attack_recall_pct','false_alarms','missed_attacks']]
        .sort_values('attack_f1_pct',ascending=False))
print('Note: OR usually catches more attacks but can create more false alarms.')

## 6. Check important attack categories and missing values

Pay special attention to crypto-ransomware (only 35 validation records), exploitation and lateral movement. These are flow-label results, not complete APT-campaign detection.

In [ ]:
category_rows=[]
for label,p in variants.items():
    for category in sorted(frame.loc[attack,'class2'].unique()):
        mask=attack & (frame['class2'].to_numpy()==category)
        total=int(mask.sum())
        category_rows.append({'candidate':label,'class2':str(category),
                              'attack_records':total,'detected':int((p[mask]==1).sum()),
                              'missed':int((p[mask]==0).sum()),
                              'recall_pct':round(100*float((p[mask]==1).mean()),2)})
category_df=pd.DataFrame(category_rows)
focus=category_df[category_df['class2'].isin(['crypto-ransomware','Exploitation','Lateral_movement'])]
display(focus[['candidate','class2','attack_records','detected','missed','recall_pct']]
        .sort_values(['class2','candidate']))

missing_rows=[]
for feat in ('Des_bytes','Duration'):
    missing_flag=frame[feat].astype(str).str.strip().str.casefold().isin(MISSING).to_numpy()
    for label,p in variants.items():
        for status,maskflag in [('missing',missing_flag),('present',~missing_flag)]:
            mask=attack & maskflag
            n=int(mask.sum())
            missing_rows.append({'candidate':label,'feature':feat,'group':status,
                                 'attack_records':n,'recall_pct':round(100*float((p[mask]==1).mean()),2) if n else np.nan})
missing_df=pd.DataFrame(missing_rows)
display(missing_df[missing_df['candidate'].isin(['LightGBM','LightGBM + RF (OR)','Three-model majority'])])

## 7. Measure ensemble laptop resource costs

Time the **complete saved pipelines**, including preprocessing, on the same laptop. RAM is for the running Python process (not model-only). All validation rows are used for batch throughput; single-flow latency uses 60 representative rows.

In [ ]:
proc=psutil.Process(os.getpid())
BENCH_RUNS=60
BENCH_NAMES=list(variants)
# Shared loaded models: combined inference executes each required pipeline sequentially.
specs={
  'LightGBM':('lightgbm',),
  'Decision Tree':('decision_tree',),
  'Random Forest':('random_forest',),
  'LightGBM + RF (OR)':('lightgbm','random_forest'),
  'LightGBM + DT (OR)':('lightgbm','decision_tree'),
  'LightGBM + RF (AND)':('lightgbm','random_forest'),
  'Three-model majority':SHORTLIST,
  'Three-model OR':SHORTLIST,
  'Three-model mean probability':SHORTLIST
}

def actual_predict(label,source):
    names=specs[label]
    if label=='Three-model mean probability':
        scores=sum(models[name].predict_proba(source)[:,1] for name in names)/len(names)
        return (scores>=0.5).astype(np.uint8)
    values=[np.asarray(models[name].predict(source),dtype=np.uint8) for name in names]
    if len(values)==1: return values[0]
    if label.endswith('(OR)') or label=='Three-model OR':
        return np.logical_or.reduce(values).astype(np.uint8)
    if label.endswith('(AND)'):
        return np.logical_and.reduce(values).astype(np.uint8)
    if label=='Three-model majority':
        return (sum(v.astype(np.int32) for v in values)>=2).astype(np.uint8)
    raise ValueError(label)

def cpu_seconds():
    t=proc.cpu_times(); return t.user+t.system

def measure_full_batch(label):
    peak=[proc.memory_info().rss]
    stop=Event()
    def sample_rss():
        while not stop.wait(.025):
            peak[0]=max(peak[0],proc.memory_info().rss)
    sampler=Thread(target=sample_rss,daemon=True)
    startcpu=cpu_seconds();start=perf_counter();sampler.start()
    try: output=actual_predict(label,X)
    finally:
        elapsed=perf_counter()-start
        cpu_delta=max(0,cpu_seconds()-startcpu)
        stop.set();sampler.join()
        peak[0]=max(peak[0],proc.memory_info().rss)
    if not np.array_equal(output,variants[label]):
        raise ValueError(f'{label}: live ensemble disagrees with cached validation predictions')
    return elapsed,100*cpu_delta/max(elapsed,1e-9),peak[0]/(1024**2)

resource_rows=[]
indices=np.linspace(0,len(X)-1,num=BENCH_RUNS,dtype=int)
for label in BENCH_NAMES:
    warm=X.iloc[[int(indices[0])]]
    for _ in range(3): actual_predict(label,warm)
    samples=[]
    for idx in indices:
        row=X.iloc[[int(idx)]]
        t0=perf_counter();actual_predict(label,row)
        samples.append((perf_counter()-t0)*1000)
    batch_seconds,cpu_pct,peak_mib=measure_full_batch(label)
    total_size=sum((MODEL_DIR/f'{name}_core19_v3.joblib').stat().st_size for name in specs[label])/(1024**2)
    resource_rows.append({
        'candidate':label,'model_count':len(specs[label]),
        'single_flow_p50_ms':round(float(np.percentile(samples,50)),3),
        'single_flow_p95_ms':round(float(np.percentile(samples,95)),3),
        'single_flow_runs':BENCH_RUNS,'validation_batch_seconds':round(batch_seconds,4),
        'batch_throughput_flows_per_second':round(len(y)/max(batch_seconds,1e-9),1),
        'process_cpu_pct_during_batch':round(cpu_pct,2),
        'peak_process_ram_mib':round(peak_mib,2),
        'combined_model_storage_mib':round(total_size,4)
    })
resource_df=pd.DataFrame(resource_rows)
display(resource_df[['candidate','model_count','single_flow_p95_ms',
                    'batch_throughput_flows_per_second','combined_model_storage_mib',
                    'peak_process_ram_mib']])
print('All resource figures are laptop measurements, not Raspberry Pi measurements.')

## 8. Save results

Save brief tables and two graphs. Choose a final ensemble only after considering F1, false alarms, latency and resource cost.

In [ ]:
comparison.to_csv(METRICS/'x_iiotid_v3_ensemble_comparison.csv',index=False)
diversity_df.to_csv(METRICS/'x_iiotid_v3_model_error_diversity.csv',index=False)
category_df.to_csv(METRICS/'x_iiotid_v3_ensemble_attack_categories.csv',index=False)
missing_df.to_csv(METRICS/'x_iiotid_v3_ensemble_missingness.csv',index=False)
resource_df.to_csv(METRICS/'x_iiotid_v3_ensemble_resources.csv',index=False)

# One clear trade-off plot.
fig,ax=plt.subplots(figsize=(9,5.5))
for r in comparison.itertuples(index=False):
    ax.scatter(r.false_positive_pct,r.attack_f1_pct,s=70)
    ax.annotate(r.candidate,(r.false_positive_pct,r.attack_f1_pct),
                xytext=(5,5),textcoords='offset points',fontsize=8)
ax.set_xlabel('False-positive rate (%)')
ax.set_ylabel('Attack F1-score (%)')
ax.set_title('X-IIoTID — Individual vs ensemble validation results')
ax.grid(alpha=.22)
fig.tight_layout()
fig.savefig(FIGURES/'x_iiotid_v3_ensemble_tradeoff.png',dpi=170,bbox_inches='tight')
plt.show();plt.close(fig)

# The single most useful diversity diagnostic: which selected model rescues LightGBM misses.
base=(attack & (lgb==0))
rescues=[int((base & (predictions[other]==1)).sum()) for other in ('decision_tree','random_forest')]
fig,ax=plt.subplots(figsize=(6,3.7))
bars=ax.bar(['Decision Tree','Random Forest'],rescues)
for bar,n in zip(bars,rescues):
    ax.text(bar.get_x()+bar.get_width()/2,bar.get_height(),str(n),ha='center',va='bottom')
ax.set_ylabel('LightGBM missed attacks rescued')
ax.set_title('Different errors on the same validation records')
ax.set_ylim(0,max(rescues+[1])*1.2)
fig.tight_layout()
fig.savefig(FIGURES/'x_iiotid_v3_ensemble_error_rescues.png',dpi=170,bbox_inches='tight')
plt.show();plt.close(fig)

summary={
    'dataset':'X-IIoTID','feature_set':'Core19','rows_train':EXPECTED['train'],
    'rows_validation':EXPECTED['validation'],'test_scored':False,
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'base_models':list(SHORTLIST),
    'candidate_count':len(variants),'single_flow_resource_benchmark_runs':BENCH_RUNS,
    'results':comparison.to_dict(orient='records'),
    'resource_results':resource_df.to_dict(orient='records'),
    'error_diversity':diversity_df.to_dict(orient='records'),
    'notes':[
        'Exploratory validation comparison; models/thresholds not evaluated on the held-out test set.',
        'Cross-model rescue is checked on identical validation row IDs.',
        'Flow-label recall is not APT campaign-level recall.',
        'Crypto-ransomware has only 35 validation samples; per-category estimates are uncertain.',
        'Dataset missingness/Service shortcuts and gateway feature extractability remain unverified.',
        'No proof of independent capture sessions or future deployment performance.',
        'Full-batch throughput and single-flow latency were measured on a laptop.',
        'Ensemble RSS is for the Python process with shared loaded models and data; not isolated model RAM.'
    ]
}
(AUDITS/'x_iiotid_v3_ensemble_screening.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
print('Saved 5 CSV tables, 2 graphs, and 1 JSON report.')
print('No model training and NO held-out test-set access.')